# 37 · 查询词权重与字段匹配

单元：S04。先修：36。

来源：[王树森固定目录](https://github.com/wangshusen/SearchEngine/tree/7339c6b126e314abdb74a89adb033d39d8e3d37b)；[Manning等《Introduction to Information Retrieval》](https://nlp.stanford.edu/IR-book/)（补充原始教材；章节1、2、6、8）。这是 Lab 自编实验；王树森后四模块只有公开目录，不称原作者完整讲义。

证据边界：构造英文库存；可解释字段打分，非BM25F完整实现。

学习方式：先预测每个对照，再只改一个变量，运行后解释差异。

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from reclab.learning.search_curriculum import *

## 输入输出与词权重
score=sum_q wq × (wt×标题词频+正文词频)。预测标题权重提高后哪些文档上升；它没有长度归一化，随后26章补BM25。

In [2]:
q='apple repair'
for w in [0.,1.,4.]: print('title weight',w,field_rank(q,title_weight=w))

title weight 0.0 [('d', 2.0), ('b', 1.0), ('c', 1.0)]
title weight 1.0 [('d', 4.0), ('a', 2.0), ('b', 2.0), ('c', 2.0)]
title weight 4.0 [('d', 10.0), ('a', 8.0), ('b', 5.0), ('c', 5.0)]


## query重复并非必选
固定文档，仅重复battery提高其软权重。对比旧BM25将query词频二值化的契约，不能把这个简单分数说成BM25。

In [3]:
for q in ['apple repair battery','apple repair battery battery battery']:
    print(q,field_rank(q,title_weight=2))

apple repair battery [('d', 7), ('a', 5), ('c', 5), ('b', 4)]
apple repair battery battery battery [('c', 9), ('d', 9), ('a', 7), ('b', 6)]


## 硬约束与软权重
要求official只能返回包含该词的文档。预测高分但缺约束的结果会被谁过滤；不可用高分补偿硬约束。

In [4]:
soft=field_rank('apple repair'); hard=field_rank('apple repair',required=['official'])
print('soft',soft,'required official',hard)
assert [d for d,s in hard]==['a']

soft [('d', 6.0), ('a', 4.0), ('b', 3.0), ('c', 3.0)] required official [('a', 4.0)]


## 误召审计
人工标签只对当前任务有效。统计precision，查看fruit spam为何获分；练习只调标题权重能否彻底修复？

In [5]:
relevant={'a','c','d'}
for w in [0.,4.]:
    order=[d for d,s in field_rank('battery repair',title_weight=w)][:2]
    print(w,order,'precision@2',len(set(order)&relevant)/2)

0.0 ['b', 'd'] precision@2 0.5
4.0 ['c', 'd'] precision@2 1.0


## 本次保存结果的解读

battery repair的标题权重从0改4，top2从b/d变为c/d，人工precision@2由0.5变1；这只对四篇构造库存成立。重复battery提升c，official硬约束只剩a，分别体现软权重和资格约束，不能把两者混用。

改参数后应以新输出重新判断，不把这段固定解读当作新的测量。

## 复盘与练习

写下输入、输出、保持不变的量与本章不能支持的结论。选一个上面的单变量参数先预测再重跑；保留失败例。回到搜索学习路径继续下一语义单元。